# Chapter 10 — Error Handling in Multi-agent Systems · Practice

**Goal:** build the error contract of a multi-agent research system layer by layer (classified API errors, structured tool errors, local recovery, subagent result envelopes, isolated fan-out, coverage annotations) and finish with a coordinator that degrades gracefully instead of failing or lying.

- Theory: [README.md](README.md). This notebook follows the same numbering (10.1 to 10.10).
- Estimated time: 2.5–3.5 hours
- Estimated API cost: **< $0.05** with the default model (`claude-sonnet-5-5`, all live cells once). Everything else runs **offline**.
- Prerequisites: [00-prerequisites](../../00-prerequisites/README.md) (venv from `requirements.txt`, `.env` with `ANTHROPIC_API_KEY`), [Chapter 2](../02-tool-use/README.md) (tool loop, `is_error`), [Chapter 3](../03-agent-sdk/README.md) (subagents, `ResultMessage`), [Chapter 8](../08-task-decomposition/README.md) (what "coverage" means)

How to read the cells:

| Marker | Meaning |
|---|---|
| **(offline)** | Runs without a key. API errors come from `ScriptedClaude`, a real `anthropic.AsyncAnthropic` client whose HTTP layer replays scripted responses, so the SDK's own exception classes, retry logic and response types run for real. Subagents and data sources are simulated with small async functions. |
| **(live)** | Calls the real API (a fraction of a cent). Without a key it prints "skipped" and moves on, so Run All works offline; no offline cell depends on a live one. |

The notebook uses top-level `await`, which Jupyter supports out of the box. This notebook does **not** run the Agent SDK or the `claude` CLI; where the Agent SDK matters (subagent envelopes, `ResultMessage`, hooks), you work with its documented data shapes offline.

In [ ]:
import asyncio
import json
import logging
import os
import random
import re
import tempfile
import time
from collections import Counter
from itertools import count
from pathlib import Path
from typing import Awaitable, Callable, Literal

import anthropic
import httpx2
from dotenv import find_dotenv, load_dotenv
from mcp import types as mcp_types
from pydantic import BaseModel, Field, ValidationError, model_validator

load_dotenv(find_dotenv())

MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-5-5")
# Async, because a coordinator runs its subagents concurrently. Explicit, modest limits per call.
client = anthropic.AsyncAnthropic(timeout=60.0, max_retries=2)   # only the (live) cells use it
print("anthropic", anthropic.__version__, "| MODEL =", MODEL)

In [ ]:
# Live cells need a key. Without one they print "skipped" and every (offline) cell still runs.
HAS_KEY = bool(os.getenv("ANTHROPIC_API_KEY"))
print("live cells enabled" if HAS_KEY else "no ANTHROPIC_API_KEY: (live) cells will be skipped")

### Offline toolkit (run this once)

`ScriptedClaude` wraps a **real** `AsyncAnthropic` client. Its transport answers with whatever you script, in order: a normal reply, an HTTP error with any status and headers, a connection reset, or a read timeout. It records every request body and header, and `attempts` counts how many HTTP requests the SDK actually made (so you can watch the SDK's retries). `FakeClock` replaces `asyncio.sleep` when we want to check backoff delays without waiting.

In [ ]:
_ids = count(1)


def text(t: str) -> dict:
    return {"type": "text", "text": t}


def tool_call(name: str, tool_input: dict, id: str | None = None) -> dict:
    return {"type": "tool_use", "id": id or f"toolu_{next(_ids):04d}", "name": name, "input": tool_input}


def reply(*blocks: dict, stop_reason: str | None = None) -> dict:
    """A Messages API response body (wire format)."""
    if stop_reason is None:
        stop_reason = "tool_use" if any(b["type"] == "tool_use" for b in blocks) else "end_turn"
    return {"id": f"msg_{next(_ids):04d}", "type": "message", "role": "assistant", "model": "scripted",
            "content": list(blocks), "stop_reason": stop_reason, "stop_sequence": None,
            "usage": {"input_tokens": 0, "output_tokens": 0}}


def api_error(status: int, err_type: str, message: str, headers: dict | None = None,
              details: dict | None = None) -> dict:
    """A scripted HTTP error in the documented shape: {"type": "error", "error": {...}, "request_id": ...}."""
    error = {"type": err_type, "message": message}
    if details is not None:
        error["details"] = details               # e.g. {"error_code": "enforced_spend_limit_reached"}
    return {"_status": status, "_headers": headers or {}, "type": "error", "error": error}


CONNECT_ERROR = "connect_error"   # sentinel: the transport raises a connection error
READ_TIMEOUT = "read_timeout"     # sentinel: the transport raises a read timeout


class ScriptedClaude:
    """Offline Messages API: replays scripted replies/errors through a real AsyncAnthropic client."""

    def __init__(self, *replies, max_retries: int = 0):
        self.replies = list(replies)
        self.requests: list[dict] = []
        self.headers: list[dict] = []
        transport = httpx2.MockTransport(self._handle)
        self.client = anthropic.AsyncAnthropic(
            api_key="offline", max_retries=max_retries,
            http_client=anthropic.DefaultAsyncHttpxClient(transport=transport),
        )

    @property
    def attempts(self) -> int:
        return len(self.requests)

    def _handle(self, request: httpx2.Request) -> httpx2.Response:
        body = json.loads(request.content)
        self.requests.append(body)
        self.headers.append(dict(request.headers))
        rid = f"req_offline_{len(self.requests):03d}"
        if not self.replies:
            return httpx2.Response(500, headers={"request-id": rid},
                                   json={"type": "error", "error": {"type": "api_error", "message": "script exhausted"}})
        item = self.replies.pop(0)
        if item == CONNECT_ERROR:
            raise httpx2.ConnectError("connection reset (scripted)", request=request)
        if item == READ_TIMEOUT:
            raise httpx2.ReadTimeout("read timed out (scripted)", request=request)
        if callable(item):
            item = item(body)          # a "fake model" that computes its reply from the request
        item = dict(item)
        status = item.pop("_status", 200)
        headers = {"request-id": rid, **item.pop("_headers", {})}
        if status >= 400:
            item["request_id"] = rid
        return httpx2.Response(status, headers=headers, json=item)


class FakeClock:
    """Drop-in for asyncio.sleep that records the delays instead of waiting."""

    def __init__(self):
        self.now, self.sleeps = 0.0, []

    async def sleep(self, seconds: float) -> None:
        self.sleeps.append(round(seconds, 3))
        self.now += seconds


print("offline toolkit ready")

## 10.1 Where failures happen in a multi-agent system

Five layers can fail: Claude API calls, tool execution, the subagent, the coordinator and the synthesis step. Each has a different owner and recovery. Before fixing anything, **isolate the origin**: an integration problem (HTTP error, truncated output) is fixed in code and configuration; a model-output problem (a wrong answer built on successful tool calls) is fixed with prompts, decomposition and validation; and a failure that was reported as success is a broken **error contract**. [Theory](README.md#101-where-failures-happen-in-a-multi-agent-system-expanded-beyond-the-guide)

In [ ]:
# (offline) Four simplified trace records, as you might pull them from your tracing tool.
TRACES = {
    "T1": {"api_error": {"status": 400, "type": "invalid_request_error"}, "stop_reason": None,
           "tool_calls": [], "unsupported_claims": 0},
    "T2": {"api_error": None, "stop_reason": "max_tokens", "tool_calls": [], "unsupported_claims": 0},
    "T3": {"api_error": None, "stop_reason": "end_turn",
           "tool_calls": [{"tool": "web_search", "raw": "200 OK, 6 hits", "reported_as": "ok"}],
           "unsupported_claims": 2},
    "T4": {"api_error": None, "stop_reason": "end_turn",
           "tool_calls": [{"tool": "patent_search", "raw": "TimeoutError after 10s", "reported_as": "empty"}],
           "unsupported_claims": 0},
}


def locate_origin(trace: dict) -> tuple[str, str]:
    """The README's origin-isolation table as code: (origin, where the fix belongs)."""
    if trace["api_error"]:
        return "integration layer", "your code / config / retries (read status, error.type, request_id)"
    if trace["stop_reason"] in ("max_tokens", "refusal"):
        return "integration / limits", "request parameters (max_tokens, schema); never ship truncated output"
    for call in trace["tool_calls"]:
        failed_for_real = not call["raw"].startswith("200")
        if failed_for_real and call["reported_as"] in ("ok", "empty"):
            return "error contract", f"{call['tool']}: '{call['raw']}' was reported as '{call['reported_as']}'"
    if trace["unsupported_claims"]:
        return "model output", "prompts, validation, human review (every tool call succeeded)"
    return "no failure found", "-"


for tid, trace in TRACES.items():
    origin, fix = locate_origin(trace)
    print(f"{tid}: {origin:<21} -> {fix}")

The integration-layer evidence comes straight from the SDK. A scripted 400 and a scripted `max_tokens` stop show what you would look at:

In [ ]:
# (offline) What the integration-layer evidence looks like in real SDK objects
sc = ScriptedClaude(
    api_error(400, "invalid_request_error",
              "messages.2: `tool_use` ids were found without `tool_result` blocks immediately after"),
    reply(text('{"findings": [{"claim": "Generative tools are used in'), stop_reason="max_tokens"),
)
try:
    await sc.client.messages.create(model=MODEL, max_tokens=300, messages=[{"role": "user", "content": "hi"}])
except anthropic.BadRequestError as err:
    print("exception :", type(err).__name__)
    print("status    :", err.status_code, "| error.type:", err.type, "| request_id:", err.request_id)
    print("message   :", err.body["error"]["message"])

r = await sc.client.messages.create(model=MODEL, max_tokens=300, messages=[{"role": "user", "content": "hi"}])
print("\nstop_reason:", r.stop_reason, "| text:", r.content[0].text)

> **Observe:** T3 and T4 both produce a polished report, but for opposite reasons. In T3 the tools worked and the model over-claimed (a model-output problem). In T4 the model did nothing wrong: it was told "no patents" when the patent database was actually unreachable. Only T4 is fixed by the error contract this chapter builds.

> **Try it:** add a T5 where every subagent succeeded but the plan never assigned "music". Which origin should it get? (Answer: none of these. It is a decomposition gap, see the exam trap in 10.1 and the plan check in 10.9.)

## 10.2 Error categories and the retryable flag

Four categories (transient, validation, business, permission) drive four different responses, and `retryable` answers one question: will the **same** request have a reasonable chance of succeeding later? The same taxonomy applies to the Claude API's own HTTP errors. The Python SDK already retries connection errors, 408, 409, 429 and 5xx twice by default (`max_retries=2`), so by the time your code catches an exception the cheap retries have happened. [Theory](README.md#102-error-categories-and-the-retryable-flag)

In [ ]:
Category = Literal["transient", "validation", "business", "permission"]


def classify_api_error(err: Exception) -> tuple[Category, bool]:
    """Map an Anthropic SDK exception to (category, retryable). The README 10.2 tables as code."""
    if isinstance(err, anthropic.APIConnectionError):     # includes APITimeoutError
        return "transient", True
    if isinstance(err, anthropic.RateLimitError):          # 429
        body = err.body if isinstance(err.body, dict) else {}
        code = (body.get("error") or {}).get("details", {}).get("error_code")
        if code == "enforced_spend_limit_reached":         # documented: the usage tier's monthly spend cap
            return "business", False
        return "transient", True                           # any other 429, with or without retry-after
    if isinstance(err, anthropic.APIStatusError):
        code = err.status_code
        if code < 400:                                      # an `error` event mid-stream, after a 200 OK
            transient = {"overloaded_error", "api_error", "timeout_error", "rate_limit_error"}
            return ("transient", True) if err.type in transient else ("validation", False)
        if code >= 500:                                     # 500, 503, 504, 529
            return "transient", True
        if code in (401, 403):
            return "permission", False
        if code == 402:
            return "business", False
        if code == 409:                                     # conflict: re-read the state, then retry
            return "transient", True
        body = err.body if isinstance(err.body, dict) else {}
        message = (body.get("error") or {}).get("message", "")
        if code == 400 and message.startswith("You have reached your specified"):
            return "business", False                       # a spend limit YOU set: no request change helps
        return "validation", False                         # 400, 404, 413, 422: fix the request
    raise err                                              # not an API error: a bug in our own code


CASES = [
    ("400 bad request", api_error(400, "invalid_request_error", "(scripted) max_tokens: field required")),
    ("400 own spend limit", api_error(400, "invalid_request_error",
                                      "You have reached your specified API usage limits. (scripted)")),
    ("401 bad key", api_error(401, "authentication_error", "(scripted) invalid x-api-key")),
    ("402 billing", api_error(402, "billing_error", "(scripted) payment problem")),
    ("403 no access", api_error(403, "permission_error", "(scripted) key lacks access to this resource")),
    ("404 not found", api_error(404, "not_found_error", "(scripted) resource not found")),
    ("409 conflict", api_error(409, "conflict_error", "(scripted) resource modified concurrently")),
    ("413 too large", api_error(413, "request_too_large", "(scripted) request exceeds 32 MB")),
    ("429 rate limit", api_error(429, "rate_limit_error", "(scripted) rate limit", {"retry-after": "7"})),
    ("429 spend cap", api_error(429, "rate_limit_error", "(scripted) monthly spend cap reached",
                                details={"error_code": "enforced_spend_limit_reached"})),
    ("429 no retry-after", api_error(429, "rate_limit_error", "(scripted) 429 with no retry-after, no error_code")),
    ("500 api_error", api_error(500, "api_error", "(scripted) internal error")),
    ("503 unavailable", api_error(503, "api_error", "(scripted) unavailable")),
    ("504 timeout", api_error(504, "timeout_error", "(scripted) timed out while processing")),
    ("529 overloaded", api_error(529, "overloaded_error", "(scripted) overloaded")),
    ("connection reset", CONNECT_ERROR),
    ("client timeout", READ_TIMEOUT),
]

sc = ScriptedClaude(*(case for _, case in CASES))       # max_retries=0: see each error once
for label, _ in CASES:
    try:
        await sc.client.messages.create(model=MODEL, max_tokens=300, messages=[{"role": "user", "content": "hi"}])
    except anthropic.APIError as err:
        category, retryable = classify_api_error(err)
        status = getattr(err, "status_code", "-")
        print(f"{label:<19} {type(err).__name__:<23} status={status!s:<4} -> {category:<10} retryable={retryable}")

> **Observe:** `503` arrives as `InternalServerError` on the first-party API (the SDK's `ServiceUnavailableError` and `DeadlineExceededError` are only raised by the Bedrock and Vertex clients), so branching on `status_code` is the portable choice. `402` has no dedicated class in `anthropic` 1.11: it arrives as a plain `APIStatusError`. `409` arrives as `ConflictError`; the SDK would normally have retried it already (we set `max_retries=0` here to see each error once).

The 429s look identical by class and status. The documented way to tell a **spend cap** from a rate limit is the body: `err.body["error"]["details"]["error_code"] == "enforced_spend_limit_reached"`. A missing `retry-after` is not proof: the spend-cap 429 never has one, but ordinary rate-limit 429s can lack it too, so "429 no retry-after" stays transient (the SDK retries every 429 as well). Likewise a 400 whose message starts "You have reached your specified ... API usage limits" is a spend limit you set yourself: business, not a request to fix.

Errors can also arrive **inside a stream**, after the API has already answered `200 OK`. The SDK raises those as a plain `APIStatusError` with `status_code == 200`, and it cannot retry them because the response had started. That is why the classifier falls back to `err.type` when the status is below 400:

In [ ]:
# (offline) an `error` event that arrives mid-stream, after a 200 OK
SSE_WITH_ERROR = (
    'event: message_start\n'
    'data: {"type": "message_start", "message": {"id": "msg_offline", "type": "message", "role": "assistant", '
    '"model": "claude-sonnet-5-5", "content": [], "stop_reason": null, "stop_sequence": null, '
    '"usage": {"input_tokens": 12, "output_tokens": 1}}}\n\n'
    'event: error\n'
    'data: {"type": "error", "error": {"type": "overloaded_error", "message": "Overloaded (scripted)"}}\n\n'
)
stream_attempts = []


def sse_handler(request: httpx2.Request) -> httpx2.Response:
    stream_attempts.append(request)
    return httpx2.Response(200, headers={"content-type": "text/event-stream", "request-id": "req_offline_stream"},
                           content=SSE_WITH_ERROR.encode())


streaming_client = anthropic.AsyncAnthropic(
    api_key="offline", max_retries=2,
    http_client=anthropic.DefaultAsyncHttpxClient(transport=httpx2.MockTransport(sse_handler)))
try:
    async with streaming_client.messages.stream(model=MODEL, max_tokens=300,
                                                messages=[{"role": "user", "content": "hi"}]) as stream:
        async for _event in stream:
            pass
except anthropic.APIStatusError as err:
    print(f"{type(err).__name__}: status_code={err.status_code} type={err.type} "
          f"attempts={len(stream_attempts)} -> {classify_api_error(err)}")

> **Observe:** one attempt only, even with `max_retries=2`, and `status_code` is `200`. A classifier that only looked at the status code would have called this a validation error.

Now watch the SDK's built-in retries. We add a tiny `retry-after-ms` header so the SDK (which honors it) does not make the demo wait for real backoff:

In [ ]:
# (offline) The SDK's own retries: count the HTTP attempts behind one messages.create() call
print("SDK defaults: max_retries =", anthropic.DEFAULT_MAX_RETRIES, "| timeout =", anthropic.DEFAULT_TIMEOUT)
fast = {"retry-after-ms": "10"}
overloaded = api_error(529, "overloaded_error", "(scripted) overloaded", fast)
ask = {"model": MODEL, "max_tokens": 300, "messages": [{"role": "user", "content": "hi"}]}

sc = ScriptedClaude(overloaded, overloaded, reply(text("Recovered.")), max_retries=2)
r = await sc.client.messages.create(**ask)
print(f"529, 529, 200         -> success after {sc.attempts} attempts: {r.content[0].text!r}")

sc = ScriptedClaude(overloaded, overloaded, overloaded, max_retries=2)
try:
    await sc.client.messages.create(**ask)
except anthropic.OverloadedError as err:
    print(f"529 x3                -> {type(err).__name__} after {sc.attempts} attempts (request_id={err.request_id})")

sc = ScriptedClaude(overloaded, max_retries=2)
try:
    await sc.client.with_options(max_retries=0).messages.create(**ask)   # per-call override
except anthropic.OverloadedError:
    print(f"529, max_retries=0    -> raised after {sc.attempts} attempt")

sc = ScriptedClaude(api_error(400, "invalid_request_error", "(scripted) bad"), max_retries=2)
try:
    await sc.client.messages.create(**ask)
except anthropic.BadRequestError:
    print(f"400                   -> raised after {sc.attempts} attempt (400 is never retried)")

> **Observe:** one `messages.create()` can be **three** HTTP requests. Keep that in mind in 10.5: if your subagent also retries three times, the counts multiply.

> **Try it:** script a `409 conflict_error` followed by a success and confirm the SDK retries it too. Then remove the `retry-after-ms` header and time the call: you will see the SDK's own backoff (about 0.5 s, then 1 s, with jitter).

In [ ]:
# (live) a tiny real call with per-call limits; an API failure is classified instead of crashing the cell
if not HAS_KEY:
    print("skipped (live): no ANTHROPIC_API_KEY")
else:
    try:
        r = await client.with_options(timeout=20.0, max_retries=1).messages.create(
            model=MODEL, max_tokens=300, messages=[{"role": "user", "content": "Reply with the single word: ready"}])
        print(r.stop_reason, "|", r.content[-1].text if r.content else "", "| request id:", r._request_id)
        print("usage:", r.usage.input_tokens, "in /", r.usage.output_tokens, "out")
    except anthropic.APIError as err:
        print(type(err).__name__, "->", classify_api_error(err), "| request id:", getattr(err, "request_id", None))

## 10.3 Access failure vs valid empty result

An **access failure** (timeout, 503, 403) means "we know nothing". A **valid empty result** means "we looked and nothing matches", which is a finding. Give each outcome its own status instead of overloading an empty list, and resolve **suspicious** empties (a degraded source that answers `200 OK` with nothing) at the lowest level, for example with a canary query that always matches. [Theory](README.md#103-access-failure-vs-valid-empty-result)

In [ ]:
class SearchOutcome(BaseModel):
    source: str
    query: str
    status: Literal["ok", "empty", "failed"]
    hits: list[dict] = Field(default_factory=list)
    error: str | None = None


async def search_source(source: str, query: str, backend, canary_query: str | None = None) -> SearchOutcome:
    """Query one source and say precisely what happened (README 10.3)."""
    try:
        hits = await backend(source, query)
    except TimeoutError:
        return SearchOutcome(source=source, query=query, status="failed", error="timeout after 10s")
    if hits:
        return SearchOutcome(source=source, query=query, status="ok", hits=hits)

    # Empty. Resolve the ambiguity HERE: is the source really answering, or silently degraded?
    if canary_query is not None:
        try:
            canary_hits = await backend(source, canary_query)   # a query that always matches
        except TimeoutError:
            canary_hits = []
        if not canary_hits:
            return SearchOutcome(source=source, query=query, status="failed",
                                 error="empty result, and a canary query that always matches also returned nothing")
    return SearchOutcome(source=source, query=query, status="empty")

In [ ]:
# (offline) Four simulated sources, four different truths (the README 10.3 example plus a degraded source)
CANARY = "any record from the last 30 days"   # a query that every healthy source in this demo matches
RULINGS = [{"title": f"Ruling {i}", "url": f"https://example.org/ruling/{i}"} for i in range(1, 7)]


async def demo_backend(source: str, query: str) -> list[dict]:
    await asyncio.sleep(0.01)                               # pretend network latency
    if source == "court_records":
        return RULINGS
    if source == "regulator":                               # healthy, nothing matches the real query
        return [{"title": "Register entry 2025-114"}] if query == CANARY else []
    if source == "news_api":
        raise TimeoutError("news API did not answer")
    if source == "trade_press":                             # degraded: 200 OK and empty for EVERY query
        return []
    raise ValueError(f"unknown source {source}")


QUERY = "AI-generated music copyright disputes 2025"
outcomes = await asyncio.gather(
    search_source("court_records", QUERY, demo_backend),
    search_source("regulator", QUERY, demo_backend, canary_query=CANARY),
    search_source("news_api", QUERY, demo_backend),
    search_source("trade_press", QUERY, demo_backend, canary_query=CANARY),
)
NEXT_STEP = {"ok": "use the hits", "empty": "report '0 matches' as a finding",
             "failed": "retry / reroute / annotate the gap"}
for o in outcomes:
    print(f"{o.source:<13} {o.status:<6} hits={len(o.hits):<3} -> {NEXT_STEP[o.status]:<34} {o.error or ''}")

> **Observe:** `regulator` and `trade_press` both returned an empty list, but the canary query separates them: `regulator` is a real "0 results", `trade_press` is a disguised outage. Without the canary the report would say "the trade press has not covered this".

> **Try it:** some search APIs report degradation in their metadata (a "failed shards" count, an "index updated at" timestamp). Change `demo_backend` to return `{"hits": [], "failed_shards": 2}` for `trade_press` and adapt `search_source` to treat failed shards as `failed`. If neither check is available, return `empty` with an `error` note such as "0 results, source health unverified".

## 10.4 Structured error responses from tools

The tool is the first place a failure is understood, so it must describe it in a way the agent can act on: `is_error: true` on a Messages API `tool_result` (or `isError: true` on an MCP result), and inside the content a category, a retry flag, a human-readable message and, for business errors, customer-safe wording plus alternatives. These field names are **your** convention: MCP only defines `isError`. [Theory](README.md#104-structured-error-responses-from-tools)

In [ ]:
def tool_error_result(tool_use_id: str, category: Category, retryable: bool, message: str, **extra) -> dict:
    """A tool_result block the model can reason about (not just 'Operation failed')."""
    payload = {"errorCategory": category, "isRetryable": retryable, "message": message, **extra}
    return {"type": "tool_result", "tool_use_id": tool_use_id, "is_error": True, "content": json.dumps(payload)}


generic = {"type": "tool_result", "tool_use_id": "toolu_01", "is_error": True, "content": "Operation failed"}
structured = tool_error_result(
    "toolu_01", "business", False,
    "Refund of $820 exceeds the $500 self-service limit.",
    customerMessage="I can't approve a refund this large myself, but I can pass it to a supervisor today.",
    alternatives=["escalate_to_human", "offer store credit up to $500"],
)
print("generic   :", generic["content"])
print("structured:", json.dumps(json.loads(structured["content"]), indent=2))

# The same failure on the MCP side, with the real MCP SDK types. Only isError is part of the protocol.
mcp_error = mcp_types.CallToolResult(content=[mcp_types.TextContent(type="text", text=structured["content"])],
                                     is_error=True)
mcp_empty = mcp_types.CallToolResult(content=[mcp_types.TextContent(type="text", text='{"orders": []}')],
                                     is_error=False)       # "no match" is a SUCCESS with an empty payload
print("\nMCP error wire shape:", json.dumps(mcp_error.model_dump(by_alias=True, exclude_none=True))[:140], "...")
print("MCP empty wire shape:", json.dumps(mcp_empty.model_dump(by_alias=True, exclude_none=True)))

Now the whole loop. Two tools on a tiny support backend: `get_order` times out once (transient), and `process_refund` enforces a $500 limit (business). The tool code raises a classified `ToolFailure`, and one function turns it into the `tool_result` the model sees.

In [ ]:
class ToolFailure(Exception):
    """A failure the tool layer has already classified (README 10.5), with optional extra payload fields."""

    def __init__(self, category: Category, retryable: bool, message: str, **extra):
        super().__init__(message)
        self.category, self.retryable, self.extra = category, retryable, extra
        self.attempts = 1


ORDERS = {"ORD-5001": {"order_id": "ORD-5001", "status": "delivered", "total_usd": 820.0}}
REFUND_LIMIT_USD = 500
flaky_calls = Counter()


def get_order(order_id: str) -> dict:
    flaky_calls["get_order"] += 1
    if flaky_calls["get_order"] == 1:
        raise ToolFailure("transient", True, "Order service timed out after 5s. The same call is safe to retry.")
    if not re.fullmatch(r"ORD-\d{4}", order_id):
        raise ToolFailure("validation", False, f"order_id {order_id!r} is malformed: expected ORD- plus 4 digits.")
    return ORDERS.get(order_id) or {"order": None, "note": "no order with this ID"}   # not found = valid empty


def process_refund(order_id: str, amount_usd: float) -> dict:
    if amount_usd > REFUND_LIMIT_USD:
        raise ToolFailure(
            "business", False, f"Refund of ${amount_usd:.0f} exceeds the ${REFUND_LIMIT_USD} self-service limit.",
            customerMessage="I can't approve a refund this large myself, but I can pass it to a supervisor today.",
            alternatives=["escalate_to_human", f"offer store credit up to ${REFUND_LIMIT_USD}"],
        )
    return {"refund_id": "RF-0001", "status": "refunded", "amount_usd": amount_usd}


HANDLERS = {"get_order": get_order, "process_refund": process_refund}
TOOLS = [
    {"name": "get_order",
     "description": "Fetch one order by ID (format ORD-#### e.g. ORD-5001). Returns status and total in USD.",
     "input_schema": {"type": "object", "properties": {"order_id": {"type": "string"}},
                      "required": ["order_id"], "additionalProperties": False}},
    {"name": "process_refund",
     "description": ("Refund a delivered order. Refunds above the self-service limit are rejected with a "
                     "business error that includes customer-safe wording and alternatives."),
     "input_schema": {"type": "object",
                      "properties": {"order_id": {"type": "string"}, "amount_usd": {"type": "number"}},
                      "required": ["order_id", "amount_usd"], "additionalProperties": False}},
]
SUPPORT_SYSTEM = (
    "You are a customer support agent. Tool errors arrive as JSON with errorCategory and isRetryable. "
    "If isRetryable is true, retry the same call once. Never retry when isRetryable is false. "
    "For business errors, explain the outcome to the customer using customerMessage and offer the alternatives. "
    "Never show internal error text to the customer."
)


def execute(block) -> dict:
    """Run one tool_use block; every outcome becomes a tool_result (success, empty or structured error)."""
    if block.name not in HANDLERS:
        return tool_error_result(block.id, "validation", False, f"Unknown tool {block.name!r}.")
    try:
        return {"type": "tool_result", "tool_use_id": block.id, "content": json.dumps(HANDLERS[block.name](**block.input))}
    except ToolFailure as f:
        return tool_error_result(block.id, f.category, f.retryable, str(f), **f.extra)


async def run_support(llm, user_text: str, max_turns: int = 6) -> tuple[str, list]:
    messages: list = [{"role": "user", "content": user_text}]
    for _ in range(max_turns):
        response = await llm.messages.create(model=MODEL, max_tokens=1024, system=SUPPORT_SYSTEM,
                                             tools=TOOLS, messages=messages)
        messages.append({"role": "assistant", "content": response.content})
        if response.stop_reason != "tool_use":
            return "".join(b.text for b in response.content if b.type == "text"), messages
        messages.append({"role": "user", "content": [execute(b) for b in response.content if b.type == "tool_use"]})
    return "(stopped: max_turns reached)", messages

In [ ]:
# (offline) The model's turns are scripted; the tool results are produced by YOUR code above
flaky_calls.clear()
sc = ScriptedClaude(
    reply(tool_call("get_order", {"order_id": "ORD-5001"})),
    reply(tool_call("get_order", {"order_id": "ORD-5001"})),            # retry: isRetryable was true
    reply(tool_call("process_refund", {"order_id": "ORD-5001", "amount_usd": 820})),
    reply(text("I can't approve a refund this large myself, but I can pass it to a supervisor today. "
               "Alternatively I can offer store credit up to $500.")),
)
answer, transcript = await run_support(sc.client, "Please refund my order ORD-5001 in full ($820).")
for msg in transcript:
    if msg["role"] == "user" and isinstance(msg["content"], list):
        for block in msg["content"]:
            payload = json.loads(block["content"])
            label = f"is_error, {payload['errorCategory']}, retryable={payload['isRetryable']}" if block.get("is_error") else "ok"
            print(f"tool_result [{label}]: {payload.get('message', payload)}")
print("\nFINAL:", answer)

> **Observe:** the model's decisions here were scripted, so the lesson is what **your code** sent back: a retryable transient error, a success, and a non-retryable business error with wording the agent can relay. Replace the business payload with `"Operation failed"` and ask yourself what a real model could possibly do with it.

Run the same loop against the real model **(live)** and check that it retries the transient error once and explains the business error instead of retrying it (this is exam exercise 1, step 3):

In [ ]:
# (live) does a real model retry the transient error and explain the business error?
if not HAS_KEY:
    print("skipped (live): no ANTHROPIC_API_KEY")
else:
    flaky_calls.clear()
    answer, transcript = await run_support(client, "Hi, please refund my order ORD-5001 in full ($820).")
    calls = [b.name for m in transcript if m["role"] == "assistant" for b in m["content"] if b.type == "tool_use"]
    print("tool calls:", calls)
    print("FINAL:", answer)

> **Try it:** (1) Call `get_order` with `"ORD-50O1"` (letter O) in a scripted turn and check that the model receives a validation error that says exactly what is wrong. (2) Change the business error to `"Operation failed"` with no fields and rerun the live cell. Count the `process_refund` calls.

## 10.5 Local recovery before propagation

Handle each error at the **lowest level that can resolve it**. Retry transient failures locally, a bounded number of times with exponential backoff and jitter; never retry validation, business or permission errors as sent; and propagate what you cannot fix, with the evidence (attempts, queries). Retry a given failure at **one** layer, and make writes idempotent before retrying them. [Theory](README.md#105-local-recovery-before-propagation)

In [ ]:
async def with_local_retry(op: Callable[[], Awaitable[list]], *, max_attempts: int = 3, base_delay: float = 0.5,
                           max_delay: float = 8.0, sleep=asyncio.sleep) -> tuple[list, int]:
    """Retry ONLY retryable failures, with capped exponential backoff and jitter (README 10.5).
    `sleep` is injectable so tests can use a fake clock. Returns (result, attempts)."""
    for attempt in range(1, max_attempts + 1):
        try:
            return await op(), attempt
        except ToolFailure as err:
            if not err.retryable or attempt == max_attempts:
                err.attempts = attempt                     # keep the evidence for the envelope
                raise
            delay = min(base_delay * 2 ** (attempt - 1), max_delay)
            await sleep(delay * random.uniform(0.5, 1.0))   # jitter avoids synchronized retry storms
    raise AssertionError("unreachable")


def scripted_op(*outcomes: str):
    """A fake backend call that follows a script: 'timeout', 'syntax', 'corrupt' or 'ok'."""
    calls: list[str] = []

    async def op():
        outcome = outcomes[min(len(calls), len(outcomes) - 1)]
        calls.append(outcome)
        if outcome == "timeout":
            raise ToolFailure("transient", True, "search backend timed out")
        if outcome == "syntax":
            raise ToolFailure("validation", False, "query syntax error near 'NEAR/3'")
        if outcome == "corrupt":
            raise ToolFailure("validation", False, "PDF is corrupted (broken xref table)")
        return [{"title": "hit"}]

    op.calls = calls
    return op


for script in [("timeout", "timeout", "ok"), ("syntax",), ("corrupt",), ("timeout",)]:
    clock, op = FakeClock(), scripted_op(*script)
    try:
        result, attempts = await with_local_retry(op, sleep=clock.sleep)
        verdict = f"recovered after {attempts} attempts"
    except ToolFailure as err:
        verdict = f"propagate {err.category} after {err.attempts} attempt(s): {err}"
    print(f"{' -> '.join(script):<26} calls={len(op.calls)} sleeps={clock.sleeps}  {verdict}")

> **Observe:** the corrupted PDF and the syntax error fail after **one** call: no backoff can fix them. A persistent timeout stops after three attempts and is propagated with `attempts=3`, which the subagent will put in its envelope.

**Retry amplification.** The SDK makes up to 3 attempts per call. Stack a subagent that retries 3 times and a coordinator that re-runs failed subagents 3 times, and one overloaded period produces:

In [ ]:
def calls_during_outage(sdk_attempts: int, subagent_attempts: int, coordinator_attempts: int) -> int:
    """HTTP requests sent for ONE task while every request fails."""
    return sdk_attempts * subagent_attempts * coordinator_attempts


print("stacked retries (3 x 3 x 3):", calls_during_outage(3, 3, 3), "requests per task")
print("one retrying layer (SDK only), coordinator changes approach once:",
      calls_during_outage(3, 1, 1) + calls_during_outage(3, 1, 1), "requests, and the second try is a different query")

**Idempotency.** A timeout on a write does not tell you whether the write happened. This fake ticket API commits the ticket and *then* loses the reply:

In [ ]:
class FakeTicketAPI:
    """Commits the write, then times out on the FIRST call, like a gateway that drops the reply."""

    def __init__(self):
        self.tickets: dict[str, dict] = {}
        self._first = True

    async def create(self, title: str, idempotency_key: str | None = None) -> dict:
        if idempotency_key and idempotency_key in self.tickets:
            return self.tickets[idempotency_key]           # same key: return the existing ticket
        ticket = {"id": f"TCK-{len(self.tickets) + 1}", "title": title}
        self.tickets[idempotency_key or ticket["id"]] = ticket   # the write HAPPENS...
        if self._first:
            self._first = False
            raise ToolFailure("transient", True, "gateway timeout after the write was committed")   # ...the reply is lost
        return ticket


for use_key in (False, True):
    api = FakeTicketAPI()
    key = "refund-ORD-5001-escalation" if use_key else None
    ticket, attempts = await with_local_retry(lambda: api.create("Refund over limit", idempotency_key=key),
                                              sleep=FakeClock().sleep)
    print(f"idempotency key={use_key!s:<5} attempts={attempts} tickets created={len(api.tickets)}")

Finally, the exam's "mixed failures inside one tool" pattern: retry timeouts **inside the tool**, and return syntax errors **immediately** with the invalid-parameter details. The model never has to decide something code can decide deterministically.

In [ ]:
async def search_tool(tool_use_id: str, query: str, backend_op, clock: FakeClock) -> dict:
    """Deterministic retry for timeouts inside the tool; immediate, specific error for bad input."""
    try:
        hits, attempts = await with_local_retry(backend_op, sleep=clock.sleep)
        return {"type": "tool_result", "tool_use_id": tool_use_id,
                "content": json.dumps({"hits": hits, "attempts": attempts})}
    except ToolFailure as err:
        return tool_error_result(tool_use_id, err.category, err.retryable, str(err),
                                 attempts=err.attempts, attemptedQuery=query)


print(await search_tool("toolu_a", "AI music", scripted_op("timeout", "ok"), FakeClock()))
print(await search_tool("toolu_b", "AI NEAR/3 music", scripted_op("syntax"), FakeClock()))

> **Try it:** write `search_with_fix(query)`: when the tool returns the `'NEAR/3'` syntax error, remove the operator and retry **once** with the corrected query (a validation retry must change something). Record both queries in an `attempted` list.

## 10.6 The structured subagent result

When a subagent cannot resolve a failure, its result must let the coordinator decide: status, failure type, category, retry flag, what was attempted, the findings it kept, alternatives and the coverage impact. A Pydantic model makes the contract explicit and **rejects** contradictory envelopes. A small `decide()` function then turns recovery into a testable policy. [Theory](README.md#106-the-structured-subagent-result)

In [ ]:
class Finding(BaseModel):
    claim: str
    source_url: str
    relevance: float


class SubagentResult(BaseModel):
    subagent: str
    topic: str
    status: Literal["success", "empty", "partial", "failed"]
    failure_type: Literal["timeout", "rate_limited", "unavailable", "invalid_input", "permission_denied",
                          "policy_blocked", "parse_error", "budget_exhausted", "internal_error"] | None = None
    error_category: Category | None = None
    retryable: bool | None = None
    attempted: list[str] = Field(default_factory=list)
    attempts: int = 1
    findings: list[Finding] = Field(default_factory=list)
    alternatives: list[str] = Field(default_factory=list)
    coverage_impact: str | None = None
    message: str | None = None

    @model_validator(mode="after")
    def consistent(self):
        failed = self.status in ("partial", "failed")
        if failed and (self.failure_type is None or self.retryable is None):
            raise ValueError("partial/failed results must say what failed and whether a retry can help")
        if not failed and self.failure_type is not None:
            raise ValueError("success/empty results must not carry a failure_type")
        if self.status == "empty" and self.findings:
            raise ValueError("an empty result cannot contain findings")
        if self.status == "partial" and not self.findings:
            raise ValueError("partial means some findings survived; use 'failed' otherwise")
        return self

In [ ]:
# (offline) The README 10.6 example envelope (copied exactly) validates; contradictory ones are rejected
example = {
    "subagent": "web_search", "topic": "film", "status": "partial",
    "failure_type": "timeout", "error_category": "transient",
    "retryable": True, "attempts": 3,
    "attempted": ["generative AI in film visual effects 2025"],
    "findings": [{"claim": "Two studios report AI-assisted rotoscoping",
                  "source_url": "https://example.org/vfx", "relevance": 0.7}],
    "alternatives": ["Narrower query: 'AI rotoscoping VFX studios'"],
    "coverage_impact": "Film: effects on VFX jobs not covered",
}
r = SubagentResult.model_validate(example)
print("valid:", r.status, r.failure_type, "retryable =", r.retryable, "|", len(r.findings), "finding kept")

one_finding = example["findings"]
bad = {
    "generic status": {"subagent": "web_search", "topic": "music", "status": "failed",
                       "message": "search unavailable"},
    "empty with findings": {"subagent": "web_search", "topic": "music", "status": "empty", "findings": one_finding},
    "partial, nothing kept": {"subagent": "web_search", "topic": "music", "status": "partial",
                              "failure_type": "timeout", "retryable": True},
    "success + failure_type": {"subagent": "web_search", "topic": "music", "status": "success",
                               "failure_type": "timeout"},
}
for label, data in bad.items():
    try:
        SubagentResult.model_validate(data)
        print(f"{label:<23} accepted?!")
    except ValidationError as err:
        print(f"{label:<23} rejected: {err.errors()[0]['msg']}")

In [ ]:
Action = Literal["accept", "retry_modified", "accept_partial", "delegate", "escalate", "annotate_gap"]


def decide(r: SubagentResult, retries_left: int, has_backup: bool) -> Action:
    """The coordinator's recovery policy (README 10.6)."""
    if r.status in ("success", "empty"):
        return "accept"                          # "no results" is a valid finding
    if r.error_category == "permission":
        return "escalate"                        # a retry cannot grant access (see Chapter 9)
    # Validation errors are not retryable AS SENT, but a CHANGED request can succeed.
    # Business-rule blocks are never retried, even when alternatives are listed.
    can_retry = r.retryable or r.error_category == "validation"
    if can_retry and retries_left > 0 and r.alternatives:
        return "retry_modified"                  # follow the specialist's suggestion, not the same query
    if r.status == "partial":
        return "accept_partial"                  # keep what survived, annotate the rest
    if has_backup:
        return "delegate"                        # another subagent or source can cover the topic
    return "annotate_gap"                        # continue without it, and say so in the report


def failed(subagent: str, ftype: str, category: Category, retryable: bool, **kw) -> SubagentResult:
    return SubagentResult(subagent=subagent, topic="music", status="failed", failure_type=ftype,
                          error_category=category, retryable=retryable, **kw)


scenarios = [
    ("empty search", SubagentResult(subagent="web_search", topic="music", status="empty"), 1, False),
    ("partial, alternatives", r, 1, False),
    ("partial, no retries left", r, 0, False),
    ("403 on patent DB", failed("patent_search", "permission_denied", "permission", False), 1, True),
    ("source down, backup exists", failed("patent_search", "unavailable", "transient", True), 1, True),
    ("source down, no backup", failed("news_search", "unavailable", "transient", True), 1, False),
    ("bad query, fix suggested", failed("news_search", "invalid_input", "validation", False,
                                        alternatives=["drop the NEAR/3 operator"]), 1, False),
    ("policy block, alternatives", failed("news_search", "policy_blocked", "business", False,
                                          alternatives=["ask a human"]), 1, False),
]
for label, result, retries_left, backup in scenarios:
    print(f"{label:<27} -> {decide(result, retries_left, backup)}")

**In the Agent SDK** a subagent returns only its **final message** to the coordinator, so the envelope must be in that message. You put the format in the subagent's prompt and validate what comes back. The wiring looks like this (not executed here, this notebook does not run the Agent SDK):

```python
from claude_agent_sdk import AgentDefinition

AgentDefinition(
    description="Searches the open web for one assigned topic.",
    prompt=WEB_SEARCH_PROMPT + ENVELOPE_INSTRUCTIONS,   # built below
    tools=["WebSearch", "WebFetch"],
    maxTurns=8,
)
```

The coordinator side is plain Python, so we can build and test it offline: generate the instructions from the Pydantic schema, then parse the **last** JSON object in the final message. A missing or invalid envelope is itself a failure, never a silent success.

In [ ]:
ENVELOPE_INSTRUCTIONS = (
    "\n\nEnd your final message with ONE JSON object matching this schema. Report timeouts and access "
    "errors as status 'failed' (or 'partial' if you kept findings), never as an empty result.\n"
    + json.dumps(SubagentResult.model_json_schema(), separators=(",", ":"))
)
print(ENVELOPE_INSTRUCTIONS.strip()[:260], "...")


def parse_final_message(subagent: str, topic: str, text: str) -> SubagentResult:
    """Validate the envelope at the end of a subagent's final message."""
    decoder, i, last = json.JSONDecoder(), 0, None
    while (i := text.find("{", i)) != -1:
        try:
            last, i = decoder.raw_decode(text, i)        # a top-level object; skip past its end
        except json.JSONDecodeError:
            i += 1
    problem = "no JSON envelope in the final message"
    if isinstance(last, dict):
        try:
            return SubagentResult.model_validate(last)
        except ValidationError as err:
            problem = f"invalid envelope: {err.errors()[0]['msg']}"
    return SubagentResult(subagent=subagent, topic=topic, status="failed", failure_type="parse_error",
                          error_category="validation", retryable=False, message=problem,
                          coverage_impact=f"{topic}: result unusable")


finals = {
    "prose + valid JSON": "I searched twice; the API timed out.\n" + json.dumps(example),
    "prose only": "Sorry, I couldn't reach the search service, so there is nothing to report.",
    "JSON missing failure_type": json.dumps({"subagent": "web_search", "topic": "music", "status": "failed"}),
}
for label, final in finals.items():
    env = parse_final_message("web_search", "music", final)
    print(f"{label:<27} -> status={env.status:<8} failure_type={env.failure_type:<10} {(env.message or '')[:60]}")

> **Observe:** "prose only" is the most dangerous case in practice. A coordinator that only reads prose might summarize it as "no music sources found". The parser turns it into an explicit `parse_error` failure instead.

> **Try it:** remove the `or r.error_category == "validation"` part of `can_retry` and rerun the scenarios above. The "bad query, fix suggested" case falls through to `annotate_gap`, so a fixable query error silently becomes a coverage gap. Then check that "policy block, alternatives" never becomes `retry_modified`, whatever you change. (Homework exercise 4 asks you to write this policy from scratch.)

## 10.7 Error-handling anti-patterns

The same timeout, handled four ways. Silent suppression and aborting the workflow are both wrong, for opposite reasons: one hides the failure, the other overreacts to it. A generic status sits in between and still leaves the coordinator guessing. [Theory](README.md#107-error-handling-anti-patterns)

In [ ]:
# (offline) The patent database is down. Four subagents report it four ways.
async def patents_backend(query: str) -> list[dict]:
    await asyncio.sleep(0.01)
    raise TimeoutError("patent DB did not answer within 10s")


async def suppressing_subagent(query: str) -> dict:
    try:
        return {"status": "success", "patents": await patents_backend(query)}
    except TimeoutError:
        return {"status": "success", "patents": []}                        # ANTI-PATTERN: silent suppression


async def generic_subagent(query: str) -> dict:
    try:
        return {"status": "success", "patents": await patents_backend(query)}
    except TimeoutError:
        return {"status": "error", "message": "patent search unavailable"}   # ANTI-PATTERN: generic status


async def aborting_subagent(query: str) -> dict:
    return {"status": "success", "patents": await patents_backend(query)}   # ANTI-PATTERN: let it escape


async def structured_subagent(query: str) -> SubagentResult:
    try:
        await patents_backend(query)
    except TimeoutError as err:
        return SubagentResult(
            subagent="patent_search", topic="music", status="failed", failure_type="timeout",
            error_category="transient", retryable=True, attempted=[query], attempts=2,
            alternatives=["narrower query: 'AI music composition patents'", "use the patent_mirror source"],
            coverage_impact="music: patent landscape not covered", message=f"{err} (2 attempts)")


def report_line(out) -> str:
    """What a coordinator can honestly write from each kind of output."""
    if isinstance(out, SubagentResult):
        return f"Patents: NOT COVERED ({out.failure_type}); next step = {decide(out, 1, True)}"
    if out["status"] == "success" and not out["patents"]:
        return "Patents: no relevant patents exist."                         # a confident falsehood
    return "Patents: search unavailable. (retry? reroute? which query? the coordinator can't tell)"


for name, sub in [("suppress", suppressing_subagent), ("generic", generic_subagent),
                  ("structured", structured_subagent)]:
    print(f"{name:<11}", report_line(await sub(QUERY)))

In [ ]:
# (offline) ANTI-PATTERN: let the exception escape inside a TaskGroup. One failure cancels the siblings.
async def good_sibling(name: str) -> str:
    await asyncio.sleep(0.05)
    return f"{name}: 7 findings"

collected: dict[str, object] = {}


async def collect(name: str, coro) -> None:
    collected[name] = await coro

try:
    async with asyncio.TaskGroup() as tg:
        tg.create_task(collect("web_search", good_sibling("web_search")))
        tg.create_task(collect("document_analysis", good_sibling("document_analysis")))
        tg.create_task(collect("patent_search", aborting_subagent(QUERY)))
except* TimeoutError as group:
    print("TaskGroup raised:", type(group).__name__, [str(e) for e in group.exceptions])
print("results that survived:", collected or "none, the two finished-in-50ms siblings were cancelled")

> **Observe:** suppression produced the most dangerous line: a confident "no relevant patents exist" with no trace of the outage. The TaskGroup run lost **all** work because one source failed.

> **Try it:** add a fifth style, "success with the error buried in metadata" (`{"status": "success", "patents": [], "meta": {"error": "timeout"}}`), and a coordinator that only checks `status`. What does `report_line` print, and why is this just suppression with extra steps?

## 10.8 Timeouts, budgets and graceful degradation

Limits at every level: per API call (`timeout`, `max_retries`), per subagent (`asyncio.timeout`, `maxTurns`), per run (deadline, `max_turns`, `max_budget_usd`), and a retry budget. And isolation: wrap each subagent so that it **always returns an envelope**, whatever happens inside. [Theory](README.md#108-timeouts-budgets-and-graceful-degradation-expanded-beyond-the-guide)

In [ ]:
def ok_result(name: str, topic: str, n: int) -> SubagentResult:
    return SubagentResult(subagent=name, topic=topic, status="success", findings=[
        Finding(claim=f"{name} finding {i}", source_url=f"https://example.org/{name}/{i}", relevance=0.7)
        for i in range(1, n + 1)])


async def web():
    await asyncio.sleep(0.05)
    return ok_result("web_search", "music", 7)

async def docs():
    await asyncio.sleep(0.05)
    return ok_result("document_analysis", "music", 3)

async def buggy():
    await asyncio.sleep(0.01)
    raise KeyError("source_url")                 # a bug, not a planned failure

async def hanging():
    await asyncio.sleep(0.6)                     # far slower than our 0.2 s deadline
    return ok_result("news_search", "music", 2)


async def run_isolated(name: str, topic: str, work: Callable[[], Awaitable[SubagentResult]],
                       deadline_s: float) -> SubagentResult:
    """Run one subagent so that its failure can never take down its siblings (README 10.8)."""
    try:
        async with asyncio.timeout(deadline_s):            # per-subagent deadline, retries included
            return await work()
    except TimeoutError:
        return SubagentResult(subagent=name, topic=topic, status="failed", failure_type="timeout",
                              error_category="transient", retryable=True,
                              message=f"no answer within {deadline_s}s", coverage_impact=f"{topic}: not covered")
    except Exception as exc:                               # a bug: none of the four categories, never retried as-is
        return SubagentResult(subagent=name, topic=topic, status="failed", failure_type="internal_error",
                              error_category=None, retryable=False, message=f"{type(exc).__name__}: {exc}",
                              coverage_impact=f"{topic}: not covered")


async def fan_out(jobs: dict[str, tuple[str, Callable[[], Awaitable[SubagentResult]]]],
                  deadline_s: float) -> list[SubagentResult]:
    return await asyncio.gather(*(run_isolated(name, topic, work, deadline_s)
                                  for name, (topic, work) in jobs.items()))


WORKS = {"web_search": web, "document_analysis": docs, "patent_search": buggy, "news_search": hanging}


async def style_taskgroup():
    async with asyncio.TaskGroup() as tg:
        tasks = [tg.create_task(w()) for w in WORKS.values()]
    return [t.result() for t in tasks]

async def style_gather_default():
    return await asyncio.gather(*(w() for w in WORKS.values()))

async def style_gather_return_exceptions():
    return await asyncio.gather(*(w() for w in WORKS.values()), return_exceptions=True)

async def style_isolated():
    return await fan_out({name: ("music", w) for name, w in WORKS.items()}, deadline_s=0.2)


for label, style in [("TaskGroup", style_taskgroup), ("gather (default)", style_gather_default),
                     ("gather(return_exceptions)", style_gather_return_exceptions),
                     ("isolated envelopes", style_isolated)]:
    t0 = time.perf_counter()
    try:
        out = await style()
        shown = [o.status if isinstance(o, SubagentResult) else f"raw {type(o).__name__}" for o in out]
    except Exception as exc:                              # an ExceptionGroup (TaskGroup) or the first raw exception (gather)
        shown = f"RAISED {type(exc).__name__}: all results lost"
    print(f"{label:<26} {time.perf_counter() - t0:4.2f}s  {shown}")

> **Observe:** only the isolated version returns four envelopes, and it does so in about 0.2 s because the hanging source is cut off by its deadline. `gather(return_exceptions=True)` keeps results but hands the coordinator a raw `KeyError` and waits for the slowest source.

**Agent SDK outcomes.** When a subagent or a whole run hits a guardrail, the Agent SDK's `ResultMessage` says so. The function below maps a `ResultMessage` (as a dict with the SDK's field names: `subtype`, `is_error`, `num_turns`, `result`, `api_error_status`, ...). `result` exists only on `success`: on an error subtype, partial work has to come from what you persisted yourself (a sink or manifest, 10.10), never from `result` to "finished" or to a failure envelope, so a cap is never presented as a finished answer.

`max_budget_usd` covers the whole run, **subagents included**: their spend counts toward the cap, and once it is reached, spawning another subagent fails with `Budget limit reached` and Claude Code stops any background subagents still running. The docs say this cap enforcement needs Claude Code v2.1.217 or later; `claude-agent-sdk` 0.2.163 bundles CLI 2.1.286 (`claude_agent_sdk._cli_version.__cli_version__`), so the bundled CLI qualifies:

In [ ]:
def run_outcome(agent: str, topic: str, result: dict) -> str | SubagentResult:
    """Treat Agent SDK guardrail hits as failures to report, not as completion."""
    subtype = result["subtype"]
    if subtype == "success" and not result.get("is_error"):
        return "finished"
    if subtype == "success":                     # is_error with an HTTP status: an API error ended the run
        status = result.get("api_error_status")
        if status == 429:
            ftype, category, retryable = "rate_limited", "transient", True
        elif status is None or status >= 500:
            ftype, category, retryable = "unavailable", "transient", True
        elif status in (401, 403):
            ftype, category, retryable = "permission_denied", "permission", False
        else:                                    # other 4xx: the request must change
            ftype, category, retryable = "invalid_input", "validation", False
        return SubagentResult(subagent=agent, topic=topic, status="failed", failure_type=ftype,
                              error_category=category, retryable=retryable,
                              message=f"API error {status if status is not None else '(no status)'}")
    guardrails = {"error_max_turns": "hit max_turns", "error_max_budget_usd": "hit max_budget_usd"}
    if subtype in guardrails:                    # no `result` here: partial work lives in your own sink/manifest
        return SubagentResult(subagent=agent, topic=topic, status="failed", failure_type="budget_exhausted",
                              error_category="business", retryable=False,
                              message=f"{guardrails[subtype]} after {result.get('num_turns')} turns")
    return SubagentResult(subagent=agent, topic=topic, status="failed", failure_type="internal_error",
                          retryable=False, message=f"{subtype}: {result.get('errors')}")


samples = [
    {"subtype": "success", "is_error": False, "num_turns": 6, "result": "Report..."},
    {"subtype": "error_max_turns", "is_error": True, "num_turns": 8},
    {"subtype": "error_max_budget_usd", "is_error": True, "num_turns": 21, "result": None},
    {"subtype": "error_during_execution", "is_error": True, "num_turns": 2, "errors": ["tool crashed"]},
    {"subtype": "success", "is_error": True, "num_turns": 3, "api_error_status": 529},
    {"subtype": "success", "is_error": True, "num_turns": 1, "api_error_status": 401},
]
for s in samples:
    out = run_outcome("web_search", "music", s)
    print(f"{s['subtype']:<23} -> {out if isinstance(out, str) else f'{out.failure_type}: {out.message}'}")

**Task budgets (beta).** On [supported models](https://platform.claude.com/docs/en/build-with-claude/task-budgets#feature-support) (Fable 5.1, Opus 5.5, Sonnet 5.5, Haiku 5.5 and some older models; not Haiku 4.5) and through the Messages API only (Claude Code and the Agent SDK don't support them), you can give Claude an **advisory** token budget for a whole agentic loop, so it wraps up with partial findings instead of being cut off. The request is built offline here to check the exact shape the SDK sends:

In [ ]:
def task_budget(total: int) -> dict:
    """output_config for an advisory loop budget. The API rejects totals below 20,000 with a 400."""
    if total < 20_000:
        raise ValueError("task_budget.total must be at least 20,000 tokens")
    return {"task_budget": {"type": "tokens", "total": total}}


sc = ScriptedClaude(reply(text("Summary of partial findings so far ...")))
await sc.client.beta.messages.create(
    model=MODEL, max_tokens=4000,                  # max_tokens stays the hard per-request cap
    output_config=task_budget(64_000), betas=["task-budgets-2026-03-13"],
    messages=[{"role": "user", "content": "Research AI in music; summarize what you found if you run low."}],
)
print("anthropic-beta header:", sc.headers[-1].get("anthropic-beta"))
print("output_config sent   :", sc.requests[-1]["output_config"])
try:
    task_budget(5_000)
except ValueError as err:
    print("too small            :", err)

**Circuit breaker and retry budget.** When a source keeps failing, stop paying for timeouts: after N consecutive failures the breaker opens and calls fail fast until a cool-down passes. A shared retry budget caps how much of a run can be spent on retries.

In [ ]:
class CircuitBreaker:
    def __init__(self, threshold: int = 3, cooldown_s: float = 30.0, clock=time.monotonic):
        self.threshold, self.cooldown_s, self.clock = threshold, cooldown_s, clock
        self.failures, self.opened_at = 0, None

    @property
    def state(self) -> str:
        if self.opened_at is None:
            return "closed"
        return "half-open" if self.clock() - self.opened_at >= self.cooldown_s else "open"

    def allow(self) -> bool:
        return self.state != "open"          # half-open lets one trial call through

    def record(self, ok: bool) -> None:
        if ok:
            self.failures, self.opened_at = 0, None
            return
        self.failures += 1
        if self.failures >= self.threshold or self.opened_at is not None:
            self.opened_at = self.clock()    # (re)open


class RetryBudget:
    def __init__(self, planned_calls: int, ratio: float = 0.2):
        self.limit, self.used = max(1, int(planned_calls * ratio)), 0

    def try_spend(self) -> bool:
        if self.used >= self.limit:
            return False
        self.used += 1
        return True


now = [0.0]
breaker = CircuitBreaker(threshold=3, cooldown_s=30, clock=lambda: now[0])
# (time, would the source answer?) - the breaker decides whether we even ask
for t, source_ok in [(0, False), (1, False), (2, False), (5, False), (20, True), (33, False), (40, True),
                     (64, True), (65, True)]:
    now[0] = t
    before = breaker.state
    if not breaker.allow():
        print(f"t={t:>3}s state={before:<9} -> fail fast with failure_type='unavailable' (no call made)")
        continue
    breaker.record(source_ok)
    print(f"t={t:>3}s state={before:<9} -> call {'ok    ' if source_ok else 'FAILED'} -> state={breaker.state}")

budget = RetryBudget(planned_calls=12)
print("\nretry budget for 12 planned calls:", budget.limit, "| grants:", [budget.try_spend() for _ in range(4)])

> **Try it:** in `style_isolated`, lower the deadline to 0.03 s. Which envelopes change, and does any sibling's result get lost? Then give `run_isolated` a `breaker` argument so it returns an `unavailable` envelope immediately when the breaker is open.

## 10.9 Coverage annotations in the final synthesis

Graceful degradation only helps if the reader can see it. Compute coverage **in code** from the envelopes and the plan (FULL, PARTIAL, NONE, with the reasons), then hand the table to the synthesis step with explicit rules: label each section, never write "no evidence exists" for a source that failed, and treat "0 results" as a finding. [Theory](README.md#109-coverage-annotations-in-the-final-synthesis)

In [ ]:
def coverage_label(results: list[SubagentResult]) -> str:
    statuses = {r.status for r in results}
    if statuses <= {"success", "empty"}:
        return "FULL"                                  # every source answered (0 results is an answer)
    if statuses & {"success", "empty", "partial"}:
        return "PARTIAL"
    return "NONE"


def coverage_table(plan: dict[str, list[str]], results: list[SubagentResult]) -> str:
    """plan maps each topic to the subagents that should have covered it (README 10.9)."""
    rows = ["| Topic | Coverage | Answered | Gaps |", "|---|---|---|---|"]
    for topic, planned in plan.items():
        got = [r for r in results if r.topic == topic]
        missing = [s for s in planned if s not in {r.subagent for r in got}]
        label = "NONE" if not got else coverage_label(got)
        if missing and label == "FULL":
            label = "PARTIAL"
        answered = [f"{r.subagent} ({'0 results' if r.status == 'empty' else len(r.findings)})"
                    for r in got if r.status != "failed"]
        gaps = [f"{r.subagent}: {r.failure_type}" for r in got if r.failure_type] + \
               [f"{s}: never ran" for s in missing]
        rows.append(f"| {topic} | {label} | {', '.join(answered) or '-'} | {'; '.join(gaps) or '-'} |")
    return "\n".join(rows)


def res(sub: str, topic: str, status: str, n: int = 0, ftype: str | None = None) -> SubagentResult:
    findings = [Finding(claim=f"{topic}: point {i} from {sub}", source_url=f"https://example.org/{sub}/{i}",
                        relevance=0.8) for i in range(1, n + 1)]
    fail = {"failure_type": ftype, "error_category": "transient", "retryable": True} if ftype else {}
    return SubagentResult(subagent=sub, topic=topic, status=status, findings=findings, **fail)


PLAN_DEMO = {"visual art": ["web_search", "documents"], "music": ["web_search", "documents"],
             "literature": ["web_search", "documents"], "film": ["web_search"]}
RESULTS_DEMO = [
    res("web_search", "visual art", "success", 7), res("documents", "visual art", "success", 3),
    res("web_search", "music", "failed", ftype="timeout"), res("documents", "music", "success", 2),
    res("web_search", "literature", "empty"), res("documents", "literature", "success", 4),
]                                                       # film's subagent never ran
print(coverage_table(PLAN_DEMO, RESULTS_DEMO))

REQUIRED_TOPICS = ["visual art", "music", "literature", "film", "games"]
never_planned = [t for t in REQUIRED_TOPICS if t not in PLAN_DEMO]
print("\nRequired but never planned (a decomposition gap, see Chapter 8):", never_planned)

> **Observe:** "film" shows up as NONE only because it was in the plan. "games" was never planned, so no coverage table can flag it: compare against the **required** topic list too.

In [ ]:
SYNTH_SYSTEM = """You write the final research report from subagent results.
Rules:
- One section per topic in <coverage>. Map the table's label to the heading suffix
  FULL -> "(FULL COVERAGE)", PARTIAL -> "(PARTIAL COVERAGE: <reason>)", NONE -> "(NO COVERAGE: <reason>)".
- State only findings that appear in <results>, each with its source URL.
- In PARTIAL and NONE sections, say in one sentence which sources were unavailable and why.
- "0 results" means the source was searched and nothing matched: report it as a finding.
  A failed source means we do not know: never write "no evidence exists" for it.
- End with a "Coverage and limitations" list."""


async def synthesize(llm, coverage_md: str, results: list[SubagentResult]) -> str:
    payload = "\n".join(r.model_dump_json() for r in results)
    response = await llm.messages.create(
        model=MODEL,
        max_tokens=4000,
        system=SYNTH_SYSTEM,
        messages=[{"role": "user", "content": f"<coverage>\n{coverage_md}\n</coverage>\n<results>\n{payload}\n</results>"}],
    )
    if response.stop_reason != "end_turn":           # max_tokens / refusal: do not ship a truncated report
        raise RuntimeError(f"Synthesis stopped with {response.stop_reason}")
    return "".join(b.text for b in response.content if b.type == "text")


# (offline) Check the request we send, and that a truncated synthesis is refused
cov = coverage_table(PLAN_DEMO, RESULTS_DEMO)
sc = ScriptedClaude(reply(text("## Report: AI impact on creative industries\n...")),
                    reply(text("## Report: AI impact on creative ind"), stop_reason="max_tokens"))
report = await synthesize(sc.client, cov, RESULTS_DEMO)
sent = sc.requests[0]
print("system starts :", sent["system"][:60], "...")
print("user content  :", sent["messages"][0]["content"][:120].replace("\n", " | "), "...")
try:
    await synthesize(sc.client, cov, RESULTS_DEMO)
except RuntimeError as err:
    print("truncated run :", err)

In [ ]:
# (live) the real synthesis step on the demo envelopes
if not HAS_KEY:
    print("skipped (live): no ANTHROPIC_API_KEY")
else:
    report = await synthesize(client, cov, RESULTS_DEMO)
    print(report)

> **Try it:** remove the line about "0 results" from `SYNTH_SYSTEM`, rerun the live cell, and read the Literature section. Does the model still distinguish "searched, nothing found" from "could not search"?

## 10.10 Observing and debugging failures

You can't improve an error contract you can't see. Log every envelope and every failed API call's `request_id`, catch tool failures as they happen (the Agent SDK's `PostToolUseFailure` hook), read failures as data (cluster by source and failure type), and persist envelopes so a crashed coordinator can **resume** instead of restarting. [Theory](README.md#1010-observing-and-debugging-failures-expanded-beyond-the-guide)

In [ ]:
class ListHandler(logging.Handler):
    """Collects formatted log lines so the notebook can show them."""

    def __init__(self):
        super().__init__()
        self.lines: list[str] = []

    def emit(self, record: logging.LogRecord) -> None:
        self.lines.append(f"{record.levelname}: {record.getMessage()}")


log = logging.getLogger("research")
log.setLevel(logging.INFO)
log.propagate = False
log_lines = ListHandler()
log.handlers = [log_lines]


# Same signature as an Agent SDK hook callback: (input_data, tool_use_id, context) -> dict
async def on_tool_failure(input_data, tool_use_id, context):
    log.warning("tool failed: %s in %s: %s", input_data["tool_name"],
                input_data.get("agent_type", "coordinator"), input_data["error"])
    return {"hookSpecificOutput": {
        "hookEventName": "PostToolUseFailure",
        "additionalContext": "If this was a timeout, retry once with a narrower query; "
                             "otherwise report it in your final JSON with failure_type and what you tried.",
    }}


# (offline) Call the hook with an input shaped like the SDK's PostToolUseFailureHookInput
fake_input = {"hook_event_name": "PostToolUseFailure", "session_id": "sess_demo", "transcript_path": "/tmp/t.jsonl",
              "cwd": "/tmp", "tool_name": "WebSearch", "tool_input": {"query": "AI music 2024"},
              "tool_use_id": "toolu_42", "error": "Request timed out after 30s", "is_interrupt": False,
              "duration_ms": 30012,          # sent by the CLI (hooks reference); not in the Python TypedDict
              "agent_type": "web_search"}
out = await on_tool_failure(fake_input, "toolu_42", None)
print(log_lines.lines[-1])
print("returned to Claude:", out["hookSpecificOutput"]["additionalContext"][:70], "...")


def log_api_failure(err: anthropic.APIError, where: str) -> None:
    category, retryable = classify_api_error(err)
    log.error("API failure in %s: %s status=%s request_id=%s -> %s, retryable=%s", where, type(err).__name__,
              getattr(err, "status_code", None), getattr(err, "request_id", None), category, retryable)


sc = ScriptedClaude(api_error(529, "overloaded_error", "(scripted) overloaded"))
try:
    await sc.client.messages.create(model=MODEL, max_tokens=300, messages=[{"role": "user", "content": "hi"}])
except anthropic.APIError as err:
    log_api_failure(err, "web_search subagent")
print(log_lines.lines[-1])

In a real Agent SDK program the hook is registered like this (not executed here):

```python
from claude_agent_sdk import ClaudeAgentOptions, HookMatcher

options = ClaudeAgentOptions(
    hooks={"PostToolUseFailure": [HookMatcher(matcher="WebSearch|WebFetch", hooks=[on_tool_failure])]},
)
```

**Read failures as data.** Here are 120 envelopes from a simulated week of runs. Cluster them, and the two very different problems separate cleanly:

In [ ]:
rng = random.Random(10)
week: list[SubagentResult] = []
for _ in range(120):
    sub = rng.choice(["web_search", "patent_search", "document_analysis"])
    roll = rng.random()
    if sub == "patent_search" and roll < 0.35:
        week.append(res(sub, "any", "failed", ftype="timeout"))               # a degrading source
    elif sub == "document_analysis" and roll < 0.25:
        week.append(SubagentResult(subagent=sub, topic="any", status="failed", failure_type="invalid_input",
                                   error_category="validation", retryable=False))   # bad tool arguments
    else:
        week.append(res(sub, "any", "success", 2))

runs = Counter(r.subagent for r in week)
failures = Counter((r.subagent, r.failure_type) for r in week if r.failure_type)
DIAGNOSIS = {"timeout": "operational: raise the timeout, add a circuit breaker, talk to the provider",
             "invalid_input": "prompt / tool design: the subagent produces bad arguments"}
for (sub, ftype), n in failures.most_common():
    print(f"{sub:<18} {ftype:<14} {n:>3}/{runs[sub]:<3} ({n / runs[sub]:.0%}) -> {DIAGNOSIS[ftype]}")

**Resume instead of restart.** Persist each envelope as it arrives (a manifest). After a crash, a new coordinator reloads the manifest and re-runs only what is missing or failed-but-retryable:

In [ ]:
class Manifest:
    """Append-only JSONL of subagent envelopes; the latest line per (subagent, topic) wins."""

    def __init__(self, path: Path):
        self.path = path

    def record(self, r: SubagentResult) -> None:
        with self.path.open("a") as f:
            f.write(r.model_dump_json() + "\n")

    def load(self) -> dict[tuple[str, str], SubagentResult]:
        done: dict[tuple[str, str], SubagentResult] = {}
        if self.path.exists():
            for line in self.path.read_text().splitlines():
                r = SubagentResult.model_validate_json(line)
                done[(r.subagent, r.topic)] = r
        return done

    def todo(self, jobs: list[tuple[str, str]]) -> list[tuple[str, str]]:
        done = self.load()
        return [j for j in jobs if j not in done or (done[j].status == "failed" and done[j].retryable)]


JOBS = [(s, t) for t in ["visual art", "music", "literature"] for s in ["web_search", "document_analysis"]]
with tempfile.TemporaryDirectory() as tmp:
    manifest = Manifest(Path(tmp) / "run-0042.jsonl")
    manifest.record(res("web_search", "visual art", "success", 3))
    manifest.record(res("document_analysis", "visual art", "success", 2))
    manifest.record(res("web_search", "music", "failed", ftype="timeout"))
    print("coordinator crashed after 3 envelopes ...")
    resumed = Manifest(Path(tmp) / "run-0042.jsonl")     # a fresh process reads the same file
    print("re-run only:", resumed.todo(JOBS))

> **Observe:** the two completed visual-art jobs are not repeated (no repeated cost, no repeated side effects), while the failed-but-retryable music search is. Chapter 11 covers manifests and state persistence in depth.

## Mini-project: a research coordinator that degrades gracefully

Everything above in one small system (exam exercise 4, step 4: simulate a subagent timeout, check that the coordinator receives structured error context, continue with partial results, annotate the gaps):

```mermaid
flowchart LR
    P["Plan: topics x subagents"] --> F["Fan-out: asyncio.gather<br/>each job isolated + deadline"]
    F --> E["Envelopes<br/>(SubagentResult)"]
    E --> D{"decide()"}
    D -->|retry_modified| F2["Re-run with the<br/>suggested query"]
    D -->|delegate| B["Backup source"]
    D -->|escalate| H["Human queue (Ch. 9)"]
    D -->|accept / accept_partial / annotate| C["coverage_table()"]
    F2 --> C
    B --> C
    C --> S["Synthesis via Messages API"]
    S --> V["verify_report(): labels match the table"]
```

Three simulated subagents (`web_search`, `document_analysis`, `patent_search`) and one backup (`patent_mirror`) query a fake backend. `FAILURE_PLAN` injects the failures. Edit it and rerun to explore. Theory: [10.6](README.md#106-the-structured-subagent-result), [10.8](README.md#108-timeouts-budgets-and-graceful-degradation-expanded-beyond-the-guide), [10.9](README.md#109-coverage-annotations-in-the-final-synthesis).

In [ ]:
TOPICS = ["visual art", "music", "literature"]
PLAN = {"visual art": ["web_search", "document_analysis", "patent_search"],
        "music": ["web_search", "document_analysis", "patent_search"],
        "literature": ["web_search", "document_analysis"]}
BACKUPS = {"patent_search": "patent_mirror"}

# Which (subagent, topic) misbehaves, and how. Edit me.
FAILURE_PLAN = {
    ("web_search", "visual art"): "flaky",              # times out once; local retry recovers
    ("web_search", "music"): "hang_after_partial",      # 1 hit, then hangs past the deadline
    ("patent_search", "visual art"): "unavailable",     # 503 every time; no query helps -> delegate
    ("patent_search", "music"): "forbidden",            # 403: service account lost access -> escalate
    ("web_search", "literature"): "empty",              # the search works; nothing matches
}
DEADLINE_S = 0.3


def broad_query(topic: str) -> str:
    return f"generative AI adoption in {topic}, 2023 to 2025"


def narrow_query(topic: str) -> str:
    return f"generative AI tools used by {topic} professionals"


class SourceBackend:
    """Fake data sources that misbehave according to the failure plan."""

    def __init__(self, failure_plan: dict):
        self.failure_plan, self.calls = failure_plan, Counter()

    async def search(self, source: str, topic: str, query: str, sink: list[Finding]) -> list[Finding]:
        key = (source, topic)
        self.calls[key] += 1
        mode = self.failure_plan.get(key, "ok")
        await asyncio.sleep(0.01)
        if mode == "flaky" and self.calls[key] == 1:
            raise ToolFailure("transient", True, "search API timed out")
        if mode == "unavailable":
            raise ToolFailure("transient", True, "503 Service Unavailable")
        if mode == "forbidden":
            raise ToolFailure("permission", False, "403: service account lacks access to the licensed patent DB")
        if mode == "empty":
            return []
        found = []
        n = 1 if (mode == "hang_after_partial" and query == broad_query(topic)) else 2
        for i in range(1, n + 1):
            f = Finding(claim=f"[{source}] evidence #{i} on AI in {topic} ({query})",
                        source_url=f"https://example.org/{source}/{topic.replace(' ', '-')}/{self.calls[key]}-{i}",
                        relevance=0.8)
            sink.append(f)                    # written as we go, so a timeout keeps it
            found.append(f)
        if mode == "hang_after_partial" and query == broad_query(topic):
            await asyncio.sleep(10)           # the source hangs; the deadline will cut us off
        return found


FAILURE_TYPES = {"transient": "unavailable", "permission": "permission_denied",
                 "validation": "invalid_input", "business": "policy_blocked"}


async def research_subagent(backend: SourceBackend, source: str, topic: str, query: str,
                            sink: list[Finding]) -> SubagentResult:
    """A simulated specialist: local recovery first, then an envelope that says exactly what happened."""
    try:
        found, attempts = await with_local_retry(lambda: backend.search(source, topic, query, sink),
                                                 max_attempts=3, base_delay=0.02)
    except ToolFailure as err:
        kept = list(sink)
        return SubagentResult(
            subagent=source, topic=topic, status="partial" if kept else "failed",
            failure_type=FAILURE_TYPES[err.category], error_category=err.category, retryable=err.retryable,
            attempted=[query], attempts=err.attempts, findings=kept,
            coverage_impact=f"{topic}: no {source} evidence", message=str(err))
    if not found:
        return SubagentResult(subagent=source, topic=topic, status="empty", attempted=[query], attempts=attempts,
                              message="search succeeded, 0 matches")
    return SubagentResult(subagent=source, topic=topic, status="success", attempted=[query], attempts=attempts,
                          findings=found, message=f"recovered after {attempts} attempts" if attempts > 1 else None)


async def run_job(backend: SourceBackend, source: str, topic: str, query: str,
                  deadline_s: float = DEADLINE_S) -> SubagentResult:
    """Isolation + deadline. A shared sink keeps partial findings when the deadline fires."""
    sink: list[Finding] = []
    try:
        async with asyncio.timeout(deadline_s):
            return await research_subagent(backend, source, topic, query, sink)
    except TimeoutError:
        return SubagentResult(
            subagent=source, topic=topic, status="partial" if sink else "failed", failure_type="timeout",
            error_category="transient", retryable=True, attempted=[query], findings=list(sink),
            alternatives=[narrow_query(topic)] if query != narrow_query(topic) else [],
            coverage_impact=f"{topic}: {source} only partly searched",
            message=f"no complete answer within {deadline_s}s; {len(sink)} finding(s) kept")
    except Exception as exc:
        return SubagentResult(subagent=source, topic=topic, status="failed", failure_type="internal_error",
                              retryable=False, attempted=[query], message=f"{type(exc).__name__}: {exc}",
                              coverage_impact=f"{topic}: no {source} evidence")


def merge(first: SubagentResult, again: SubagentResult) -> SubagentResult:
    """Combine an original partial result with the result of a modified retry; nothing is thrown away."""
    data = again.model_dump()
    data["findings"] = [f.model_dump() for f in first.findings] + data["findings"]
    data["attempted"] = first.attempted + again.attempted
    data["attempts"] = first.attempts + again.attempts
    if data["status"] == "empty" and data["findings"]:
        data["status"] = "success"
    if data["status"] == "failed" and data["findings"]:
        data["status"] = "partial"
    data["message"] = f"first try: {first.message}; retry with modified query: {again.message or again.status}"
    return SubagentResult.model_validate(data)


async def coordinate(plan: dict[str, list[str]], failure_plan: dict, *, retry_budget: int = 1,
                     deadline_s: float = DEADLINE_S):
    backend = SourceBackend(failure_plan)
    jobs = [(source, topic) for topic, sources in plan.items() for source in sources]
    first_pass = await asyncio.gather(*(run_job(backend, s, t, broad_query(t), deadline_s) for s, t in jobs))
    retries_left, final, decisions, escalations = retry_budget, [], [], []
    for r in first_pass:
        action = decide(r, retries_left, has_backup=r.subagent in BACKUPS)
        decisions.append((r.subagent, r.topic, r.status, r.failure_type, r.attempts, action))
        if action == "retry_modified":
            retries_left -= 1
            r = merge(r, await run_job(backend, r.subagent, r.topic, r.alternatives[0], deadline_s))
        elif action == "delegate":
            backup = await run_job(backend, BACKUPS[r.subagent], r.topic, r.attempted[0], deadline_s)
            r = backup.model_copy(update={"subagent": r.subagent,
                                          "message": f"served by backup {BACKUPS[r.subagent]}; primary: {r.message}"})
        elif action == "escalate":
            escalations.append({"topic": r.topic, "subagent": r.subagent, "reason": r.message})
        final.append(r)
    return final, decisions, escalations, backend


t0 = time.perf_counter()
results, decisions, escalations, backend = await coordinate(PLAN, FAILURE_PLAN)
print(f"coordinator finished in {time.perf_counter() - t0:.2f}s with {sum(backend.calls.values())} backend calls\n")
print(f"{'topic':<11} {'subagent':<18} {'status':<8} {'failure_type':<18} attempts  decision")
for sub, topic, status, ftype, attempts, action in decisions:
    print(f"{topic:<11} {sub:<18} {status:<8} {ftype or '-':<18} {attempts:^8}  -> {action}")
print("\nescalations:", escalations)

In [ ]:
# The envelope the coordinator actually received for the music web search (structured error context)
music_web = next(r for r in results if (r.subagent, r.topic) == ("web_search", "music"))
print(music_web.model_dump_json(indent=2, exclude_defaults=True))

coverage_md = coverage_table(PLAN, results)
print("\n" + coverage_md)
missing = [t for t in TOPICS if t not in PLAN]
print("\nrequired topics never planned:", missing or "none")

Now the synthesis. Offline, a small "fake model" follows `SYNTH_SYSTEM` mechanically so the whole pipeline runs without a key. Then `verify_report()` checks, **in code**, that every section's heading carries the label the coverage table computed. You would keep that check in production: the model writes the prose, but the coverage labels are facts the code already knows.

In [ ]:
def parse_table(coverage_md: str) -> list[list[str]]:
    return [[c.strip() for c in line.strip("|").split("|")] for line in coverage_md.splitlines()[2:]]


def fake_synthesis_model(body: dict) -> dict:
    """Plays the synthesis model offline by applying SYNTH_SYSTEM's rules literally."""
    user = body["messages"][0]["content"]
    cov = user.split("<coverage>\n")[1].split("\n</coverage>")[0]
    found = [json.loads(line) for line in user.split("<results>\n")[1].split("\n</results>")[0].splitlines()]
    out, limits = ["## Report: AI impact on creative industries", "", cov, ""], []
    for topic, label, _answered, gaps in parse_table(cov):
        suffix = {"FULL": "(FULL COVERAGE)", "PARTIAL": f"(PARTIAL COVERAGE: {gaps})",
                  "NONE": f"(NO COVERAGE: {gaps})"}[label]
        out.append(f"### {topic.capitalize()} {suffix}")
        for r in (r for r in found if r["topic"] == topic):
            out += [f"- {f['claim']} ({f['source_url']})" for f in r["findings"][:2]]
            if r["status"] == "empty":
                out.append(f"- {r['subagent']} searched and found 0 matching results.")
        if label != "FULL":
            out.append(f"Note: some sources were unavailable ({gaps}); conclusions here rest on the rest.")
            limits.append(f"- {topic}: {gaps}")
        out.append("")
    out += ["### Coverage and limitations", *(limits or ["- none"])]
    return reply(text("\n".join(out)))


def verify_report(report: str, coverage_md: str) -> list[str]:
    """Deterministic guard: each topic heading must carry the label computed in code."""
    marker = {"FULL": "(FULL COVERAGE", "PARTIAL": "(PARTIAL COVERAGE", "NONE": "(NO COVERAGE"}
    headings = [line.lower() for line in report.splitlines() if line.startswith("#")]
    problems = []
    for topic, label, *_ in parse_table(coverage_md):
        if not any(topic.lower() in h and marker[label].lower() in h for h in headings):
            problems.append(f"'{topic}' heading is missing '{marker[label]})'")
    if "no evidence exists" in report.lower():
        problems.append("report claims 'no evidence exists'")
    return problems


fake_llm = ScriptedClaude(fake_synthesis_model)
report = await synthesize(fake_llm.client, coverage_md, results)
print(report)
print("\nverify_report:", verify_report(report, coverage_md) or "OK")

In [ ]:
# (live) same envelopes, real synthesis model, same deterministic check
if not HAS_KEY:
    print("skipped (live): no ANTHROPIC_API_KEY")
else:
    report = await synthesize(client, coverage_md, results)
    print(report)
    print("\nverify_report:", verify_report(report, coverage_md) or "OK")

> **Observe:** the coordinator never raised and never lied. Visual art recovered twice (the web search's local retry shows as `attempts=2`; the patent source was delegated to its backup), music kept its partial web finding **and** got two more from the narrower query, the patent permission problem went to a human, and literature's "0 results" is reported as a finding. The only PARTIAL section is the one with a real, explained gap.

> **Try it:**
> 1. `coordinate(PLAN, FAILURE_PLAN, retry_budget=0)`: music's web search should become `accept_partial`, and the report should still carry its single finding.
> 2. Set `("document_analysis", "music"): "unavailable"`: there is no backup for that subagent, so you should see `annotate_gap`.
> 3. Add `"film": ["web_search"]` to `PLAN` with `("web_search", "film"): "unavailable"` and check that the report gets a `(NO COVERAGE: ...)` section.
> 4. Make `SourceBackend.search` raise a `KeyError` for one source and confirm that `run_job` turns the bug into an `internal_error` envelope while every other job still completes.

## Recap

- **Locate the layer first**: integration errors (status, `request_id`, `stop_reason`) are fixed in code; wrong answers on successful tools are model-output problems; a failure reported as success is a broken error contract.
- **Classify every error** (transient, validation, business, permission) and set `retryable` to match. The SDK already retries connection errors, 408, 409, 429 and 5xx twice; don't stack more retries on top of it.
- **"0 results" is a finding, a timeout is not.** Give them different statuses, and resolve suspicious empties locally (canary query, health metadata).
- **Tools return structured errors**: `is_error` / `isError` plus category, retry flag, a specific message, and for business errors customer-safe wording and alternatives.
- **Recover locally, propagate the rest with context**: bounded retries with backoff and jitter only for retryable errors, idempotent writes, then an envelope with failure type, attempts, partial findings and alternatives.
- **The coordinator decides with a policy** (accept, retry modified, accept partial, delegate, escalate, annotate), and **isolates** each subagent behind an envelope and a deadline, so one failure never cancels the rest.
- **Coverage is computed in code** from the envelopes and the plan, and the synthesis must show it. Check required topics against the plan too.
- **Observe**: log envelopes and `request_id`s, cluster failures by source and type, persist a manifest so you can resume.

Next: [homework](02_homework.ipynb)